In [1]:
print("ok!")


ok!


In [2]:
print("i am ok")

i am ok


In [3]:
import importlib.metadata as md

for pkg in [
    "langchain",
    "langchain-core",
    "langchain-community",
    "langchain-pinecone"
]:
    try:
        print(pkg, md.version(pkg))
    except Exception:
        print(pkg, "Not installed")

langchain 0.1.20
langchain-core 0.1.53
langchain-community 0.0.38
langchain-pinecone 0.1.1


In [4]:
import importlib.metadata as md

print("ctransformers ->", md.version("ctransformers"))

ctransformers -> 0.2.27


In [ ]:
from langchain.chains import RetrievalQA

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Pinecone

from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.llms import CTransformers

import pinecone
from pinecone import ServerlessSpec
from langchain.prompts import PromptTemplate

In [ ]:
PINECONE_API_KEY="pcsk_5TqHjy_2upVuzEZ6R8wHbwsSfJasypzSyCMWWuEGQRyNTkgzHred3rZDD27zm3Bd5cKPnF"
PINECONE_API_ENV="https://medical-chatbot-cx8aset.svc.aped-4627-b74a.pinecone.io"


In [7]:
def load_pdf(data):
    loader= DirectoryLoader(data, glob="*.pdf", loader_cls=PyPDFLoader)
    documents=loader.load()
    return documents

In [8]:
extracted_data=load_pdf("data/")

In [11]:
def text_split(extr_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extr_data)
    return text_chunks

In [12]:
text_chunks_=text_split(extracted_data)
print("length of my chunks :",len(text_chunks_))

length of my chunks : 7020


In [16]:
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [17]:
embeddingg=download_hugging_face_embeddings()

In [18]:
embeddingg

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [19]:
query_result=embeddingg.embed_query("hello world")
print("length:", len(query_result))

length: 384


In [20]:
query_result

[-0.034477293491363525,
 0.031023185700178146,
 0.006734919268637896,
 0.026108955964446068,
 -0.03936200588941574,
 -0.16030244529247284,
 0.06692398339509964,
 -0.00644145580008626,
 -0.047450482845306396,
 0.014758873730897903,
 0.07087531685829163,
 0.05552761256694794,
 0.019193336367607117,
 -0.026251327246427536,
 -0.010109526105225086,
 -0.026940451934933662,
 0.02230745740234852,
 -0.02222668007016182,
 -0.14969263970851898,
 -0.017492998391389847,
 0.007676251698285341,
 0.05435226485133171,
 0.003254401497542858,
 0.031725890934467316,
 -0.08462139964103699,
 -0.029405971989035606,
 0.051595598459243774,
 0.04812406003475189,
 -0.003314854810014367,
 -0.05827920511364937,
 0.04196922481060028,
 0.022210687398910522,
 0.1281888335943222,
 -0.02233893983066082,
 -0.011656275019049644,
 0.06292839348316193,
 -0.032876357436180115,
 -0.0912260189652443,
 -0.03117534890770912,
 0.05269956961274147,
 0.04703487083315849,
 -0.08420306444168091,
 -0.030056191608309746,
 -0.020744830

In [21]:
import os
from dotenv import load_dotenv

from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_community.embeddings import HuggingFaceEmbeddings


In [22]:
load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

# Embedding model
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Connect to Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medical-chatbot"

# Create the index if it doesn't exist
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,  # all-MiniLM-L6-v2 embedding size
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

# Connect to the index
index = pc.Index(index_name)

# Store documents
vectorstore = PineconeVectorStore.from_documents(
    documents=text_chunks_,
    embedding=embedding,
    index_name=index_name
)

print("Index created successfully!")

Index created successfully!


In [23]:
docsearch = PineconeVectorStore(
    index_name=index_name,
    embedding=embedding
)

query = "What are Allergies"

docs = docsearch.similarity_search(
    query=query,
    k=3
)

print("Result:", docs)

Result: [Document(page_content="GALE ENCYCLOPEDIA OF MEDICINE 2 117Allergies\nAllergic rhinitis is commonly triggered by\nexposure to household dust, animal fur,or pollen. The foreign substance thattriggers an allergic reaction is calledan allergen.\nThe presence of an allergen causes the\nbody's lymphocytes to begin producingIgE antibodies. The lymphocytes of an allergy sufferer produce an unusuallylarge amount of IgE.\nIgE molecules attach to mast\ncells, which contain histamine.HistaminePollen grains\nLymphocyte\nFIRST EXPOSURE", metadata={'page': 130.0, 'source': 'data\\Medical_book.pdf'}), Document(page_content='allergens are the following:\n• plant pollens\n• animal fur and dander\n• body parts from house mites (microscopic creatures\nfound in all houses)\n• house dust• mold spores• cigarette smoke• solvents• cleaners\nCommon food allergens include the following:\n• nuts, especially peanuts, walnuts, and brazil nuts\n• fish, mollusks, and shellfish• eggs• wheat• milk• food additi

In [26]:
for i, doc in enumerate(docs, 1):
    print(f"\nDocument {i}")
    print(doc.page_content)
    print("-" * 80)


Document 1
GALE ENCYCLOPEDIA OF MEDICINE 2 117Allergies
Allergic rhinitis is commonly triggered by
exposure to household dust, animal fur,or pollen. The foreign substance thattriggers an allergic reaction is calledan allergen.
The presence of an allergen causes the
body's lymphocytes to begin producingIgE antibodies. The lymphocytes of an allergy sufferer produce an unusuallylarge amount of IgE.
IgE molecules attach to mast
cells, which contain histamine.HistaminePollen grains
Lymphocyte
FIRST EXPOSURE
--------------------------------------------------------------------------------

Document 2
allergens are the following:
• plant pollens
• animal fur and dander
• body parts from house mites (microscopic creatures
found in all houses)
• house dust• mold spores• cigarette smoke• solvents• cleaners
Common food allergens include the following:
• nuts, especially peanuts, walnuts, and brazil nuts
• fish, mollusks, and shellfish• eggs• wheat• milk• food additives and preservatives
The follo

In [27]:
prompt_template="""
Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: {context}
Question: {question}

Only return the helpful answer below and nothing else.
Helpful answer:
"""

PROMPT=PromptTemplate(template=prompt_template, input_variables=["context","question"])
chain_type_kwargs ={"prompt": PROMPT}

In [28]:
llm = CTransformers(
    model="model/llama-2-7b-chat.ggmlv3.q3_K_S.bin",
    model_type="llama",
    config={
        "max_new_tokens": 512,
        "temperature": 0.8
    }
)

In [29]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever(search_kwargs={"k": 2}),
    return_source_documents=True,
    chain_type_kwargs=chain_type_kwargs
)

In [31]:
while True:
    user_input = input("Input Prompt: ")

    if user_input.lower() in ["exit", "quit"]:
        break

    result = qa.invoke({"query": user_input})

    print("Response:", result["result"])

Response: Heart attack, also known as myocardial infarction (MI), occurs when the blood flow to a part of the heart is blocked, causing damage or death to the heart muscle. This can happen due to a blockage in one of the coronary arteries, which supplies blood to the heart muscle. The blockage can be caused by a build-up of plaque in the arteries, which can lead to a heart attack. Symptoms of a heart attack can include chest pain or discomfort, shortness of breath, and lightheadedness or fainting. If you suspect that you or someone else is having a heart attack, it is important to seek medical attention immediately.


In [ ]:
#checkin pinecone still active or not
from pinecone import Pinecone
import os
from dotenv import load_dotenv

load_dotenv()

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

print(pc.list_indexes().names())

['medical-chatbot']
